# DeepLense Test 5 - Lens vs Non-Lens Classification

## Objective
Train a binary classifier on `train_lenses` / `train_nonlenses` and evaluate it on the held-out `test_lenses` / `test_nonlenses` split.

## Submission Strategy
- Use a compact residual CNN on the 3 x 64 x 64 inputs.
- Handle class imbalance with class-weighted binary cross-entropy (`pos_weight = N_neg / N_pos`).
- Use a stratified validation split and choose checkpoints by validation AUROC.
- Report ROC curve and AUROC as the official task metric.
- Report AP / PR curves as supplementary imbalance-aware diagnostics.
- Apply D4 test-time augmentation (rotations + reflections) at evaluation time and compare it against single-view inference.


In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

plt.style.use('seaborn-v0_8-whitegrid')

# Reproducibility and experiment configuration
SEED = 42
VAL_SIZE = 0.1
BATCH_SIZE = 256
EPOCHS = 80
LR = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 5
NUM_WORKERS = 4
BEST_MODEL_PATH = 'best_model.pt'


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'

# Resolve paths so the notebook works both from the repo root and from test5/
cwd = Path.cwd()
ROOT = cwd if (cwd / 'dataset').exists() else cwd / 'test5'
DATA_ROOT = ROOT / 'dataset'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f'Device: {device}')
print(f'Data root: {DATA_ROOT}')
print(f'Artifacts: {ARTIFACTS}')


## Data And Preprocessing
The dataset contains `.npy` tensors with shape `(3, 64, 64)`. The pipeline below:
- builds a stratified validation split from the provided training data,
- computes per-channel normalization on the training split only,
- applies simple geometric augmentations that are natural for this astronomy task.


In [ ]:
# Load paths and create a stratified train / validation split.
def load_class(folder_name, label):
    paths = sorted((DATA_ROOT / folder_name).glob('*.npy'))
    if not paths:
        raise FileNotFoundError(f'No .npy files in {DATA_ROOT / folder_name}')
    return [(str(path), label) for path in paths]


train_data = load_class('train_nonlenses', 0) + load_class('train_lenses', 1)
test_data = load_class('test_nonlenses', 0) + load_class('test_lenses', 1)

train_paths, train_labels = zip(*train_data)
test_paths, test_labels = zip(*test_data)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    list(train_paths),
    list(train_labels),
    test_size=VAL_SIZE,
    stratify=train_labels,
    random_state=SEED,
)

split_stats = pd.DataFrame(
    [
        {'split': 'train', 'samples': len(train_labels), 'positives': int(sum(train_labels))},
        {'split': 'val', 'samples': len(val_labels), 'positives': int(sum(val_labels))},
        {'split': 'test', 'samples': len(test_labels), 'positives': int(sum(test_labels))},
    ]
)
split_stats['negative'] = split_stats['samples'] - split_stats['positives']
split_stats['positive_fraction'] = split_stats['positives'] / split_stats['samples']
print(split_stats.round(4).to_string(index=False))


In [ ]:
# Compute channel-wise normalization statistics on the training split only.
channel_sum = np.zeros(3)
channel_sq = np.zeros(3)
total_pixels = 0

for path in train_paths:
    image = np.load(path).astype(np.float32)
    flat = image.reshape(3, -1)
    channel_sum += flat.sum(axis=1)
    channel_sq += (flat ** 2).sum(axis=1)
    total_pixels += flat.shape[1]

mean = channel_sum / total_pixels
std = np.sqrt(channel_sq / total_pixels - mean ** 2)

print(f'Mean (RGB): {mean}')
print(f'Std (RGB): {std}')


In [ ]:
# Dataset and data loaders
class LensDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = torch.from_numpy(np.load(self.paths[index]).astype(np.float32))
        if self.transform is not None:
            image = self.transform(image)
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return image, label


class RandomRotate90:
    """Rotate by multiples of 90 degrees without interpolation blur."""

    def __call__(self, image):
        k = torch.randint(0, 4, ()).item()
        return torch.rot90(image, k, dims=(-2, -1))


# Train-time augmentation mirrors the approximate rotational / reflection symmetries
# of the lens detection task while keeping the data physically plausible.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    RandomRotate90(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])

eval_transform = transforms.Normalize(mean.tolist(), std.tolist())

train_loader = DataLoader(
    LensDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    LensDataset(val_paths, val_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    LensDataset(test_paths, test_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)


## Model And Training
A lightweight ResNet-style CNN is used as a strong baseline for small 64 x 64 images. The key design choices are:
- class-weighted BCE for imbalance handling,
- AdamW optimization,
- validation AUROC for checkpoint selection and LR scheduling.


In [ ]:
# Model definition
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        out = torch.nn.functional.silu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.nn.functional.silu(out + self.shortcut(x))


class LensBinaryResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.layer2 = nn.Sequential(ResidualBlock(32, 64, stride=2), ResidualBlock(64, 64))
        self.layer3 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.layer4 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 1))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)


model = LensBinaryResNet().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'Parameters: {parameter_count:,}')


In [ ]:
# Training helpers
@torch.no_grad()
def predict_probabilities(model, loader, tta_fn=None):
    model.eval()
    all_labels, all_probs = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=pin_memory)

        if tta_fn is None:
            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)
        else:
            # Average probabilities over multiple symmetry-preserving test-time views.
            probs = torch.stack(
                [torch.sigmoid(model(view).squeeze(1)) for view in tta_fn(images)],
                dim=0,
            ).mean(dim=0)

        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

    y_true = torch.cat(all_labels).numpy()
    y_prob = torch.cat(all_probs).numpy()
    return y_true, y_prob


def summarize_predictions(y_true, y_prob):
    return {
        'auroc': roc_auc_score(y_true, y_prob),
        'ap': average_precision_score(y_true, y_prob),
        'labels': y_true,
        'probs': y_prob,
    }


def evaluate(model, loader, tta_fn=None):
    y_true, y_prob = predict_probabilities(model, loader, tta_fn=tta_fn)
    return summarize_predictions(y_true, y_prob)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=pin_memory)
        labels = labels.to(device, non_blocking=pin_memory)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(images)

    return running_loss / len(loader.dataset)


# Class weighting is the primary imbalance-mitigation strategy in this baseline.
neg_count = len(train_labels) - sum(train_labels)
pos_count = sum(train_labels)
pos_weight = torch.tensor([neg_count / pos_count], device=device)
print(f'Positive class weight: {pos_weight.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_score = -np.inf
best_state = copy.deepcopy(model.state_dict())
best_summary = None
patience = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_results = evaluate(model, val_loader)
    scheduler.step(val_results['auroc'])
    lr = optimizer.param_groups[0]['lr']

    history.append(
        {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_auroc': val_results['auroc'],
            'val_ap': val_results['ap'],
            'lr': lr,
        }
    )

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
        f"val_AUROC={val_results['auroc']:.4f} val_AP={val_results['ap']:.4f} | "
        f"lr={lr:.2e}"
    )

    current_score = float(val_results['auroc'])
    if current_score > best_score:
        best_score = current_score
        best_state = copy.deepcopy(model.state_dict())
        best_summary = {
            'epoch': epoch,
            'auroc': float(val_results['auroc']),
            'ap': float(val_results['ap']),
        }
        torch.save(
            {'model': best_state, 'history': history, 'best_summary': best_summary},
            ARTIFACTS / BEST_MODEL_PATH,
        )
        patience = 0
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            print('Early stopping')
            break

model.load_state_dict(best_state)
print(f"Best validation epoch: {best_summary['epoch']}")
print(f"Best val AUROC: {best_summary['auroc']:.4f}")
print(f"Best val AP: {best_summary['ap']:.4f}")


In [ ]:
# Training curves
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['val_auroc'], label='val AUROC')
axes[1].plot(history_df['epoch'], history_df['val_ap'], label='val AP')
axes[1].set_xlabel('Epoch')
axes[1].set_title('Validation Metrics')
axes[1].legend()

plt.tight_layout()
plt.show()


## Evaluation And Test-Time Augmentation
The official score is reported with ROC / AUROC. To make the final evaluation stronger and more stable, the notebook also evaluates D4 test-time augmentation, which averages predictions across the eight dihedral symmetries of a square image.


In [ ]:
# D4 test-time augmentation and final evaluation
def d4_tta_views(images):
    # D4 = four rotations plus four reflected counterparts.
    transposed = images.transpose(-2, -1)
    views = [
        images,
        torch.rot90(images, 1, dims=(-2, -1)),
        torch.rot90(images, 2, dims=(-2, -1)),
        torch.rot90(images, 3, dims=(-2, -1)),
        transposed,
        torch.rot90(transposed, 1, dims=(-2, -1)),
        torch.rot90(transposed, 2, dims=(-2, -1)),
        torch.rot90(transposed, 3, dims=(-2, -1)),
    ]
    return [view.contiguous() for view in views]


runs = {
    'Validation / no TTA': evaluate(model, val_loader),
    'Test / no TTA': evaluate(model, test_loader),
    'Validation / D4 TTA': evaluate(model, val_loader, tta_fn=d4_tta_views),
    'Test / D4 TTA': evaluate(model, test_loader, tta_fn=d4_tta_views),
}

summary_rows = []
for name, results in runs.items():
    split, setting = name.split(' / ')
    summary_rows.append(
        {
            'split': split,
            'setting': setting,
            'auroc': results['auroc'],
            'ap': results['ap'],
        }
    )

summary_df = pd.DataFrame(summary_rows)
print('Official task metric: ROC curve and AUROC on the held-out test set.\n')
print(summary_df.round(4).to_string(index=False))

val_delta_auroc = runs['Validation / D4 TTA']['auroc'] - runs['Validation / no TTA']['auroc']
val_delta_ap = runs['Validation / D4 TTA']['ap'] - runs['Validation / no TTA']['ap']
test_delta_auroc = runs['Test / D4 TTA']['auroc'] - runs['Test / no TTA']['auroc']
test_delta_ap = runs['Test / D4 TTA']['ap'] - runs['Test / no TTA']['ap']

print('\nD4 TTA deltas:')
print(f'  Validation: ΔAUROC={val_delta_auroc:+.4f}, ΔAP={val_delta_ap:+.4f}')
print(f'  Test: ΔAUROC={test_delta_auroc:+.4f}, ΔAP={test_delta_ap:+.4f}')

recommended_submission = runs['Test / D4 TTA']
print(
    f'\nRecommended submission setting: D4 TTA | '
    f"test AUROC={recommended_submission['auroc']:.4f}, "
    f"test AP={recommended_submission['ap']:.4f}"
)


In [ ]:
# ROC and PR curves: dashed = single-view inference, solid = D4 TTA
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

val_fpr, val_tpr, _ = roc_curve(runs['Validation / no TTA']['labels'], runs['Validation / no TTA']['probs'])
test_fpr, test_tpr, _ = roc_curve(runs['Test / no TTA']['labels'], runs['Test / no TTA']['probs'])
val_fpr_tta, val_tpr_tta, _ = roc_curve(runs['Validation / D4 TTA']['labels'], runs['Validation / D4 TTA']['probs'])
test_fpr_tta, test_tpr_tta, _ = roc_curve(runs['Test / D4 TTA']['labels'], runs['Test / D4 TTA']['probs'])

axes[0].plot(val_fpr, val_tpr, '--', alpha=0.7, label=f'Val / no TTA (AUC={runs["Validation / no TTA"]["auroc"]:.3f})')
axes[0].plot(test_fpr, test_tpr, '--', alpha=0.7, label=f'Test / no TTA (AUC={runs["Test / no TTA"]["auroc"]:.3f})')
axes[0].plot(val_fpr_tta, val_tpr_tta, label=f'Val / D4 TTA (AUC={runs["Validation / D4 TTA"]["auroc"]:.3f})')
axes[0].plot(test_fpr_tta, test_tpr_tta, label=f'Test / D4 TTA (AUC={runs["Test / D4 TTA"]["auroc"]:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

val_prec, val_rec, _ = precision_recall_curve(runs['Validation / no TTA']['labels'], runs['Validation / no TTA']['probs'])
test_prec, test_rec, _ = precision_recall_curve(runs['Test / no TTA']['labels'], runs['Test / no TTA']['probs'])
val_prec_tta, val_rec_tta, _ = precision_recall_curve(runs['Validation / D4 TTA']['labels'], runs['Validation / D4 TTA']['probs'])
test_prec_tta, test_rec_tta, _ = precision_recall_curve(runs['Test / D4 TTA']['labels'], runs['Test / D4 TTA']['probs'])

axes[1].plot(val_rec, val_prec, '--', alpha=0.7, label=f'Val / no TTA (AP={runs["Validation / no TTA"]["ap"]:.3f})')
axes[1].plot(test_rec, test_prec, '--', alpha=0.7, label=f'Test / no TTA (AP={runs["Test / no TTA"]["ap"]:.3f})')
axes[1].plot(val_rec_tta, val_prec_tta, label=f'Val / D4 TTA (AP={runs["Validation / D4 TTA"]["ap"]:.3f})')
axes[1].plot(test_rec_tta, test_prec_tta, label=f'Test / D4 TTA (AP={runs["Test / D4 TTA"]["ap"]:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

plt.tight_layout()
plt.show()
